Load Packages

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

Load data from directory 

In [2]:
file_path = Path("Data/raw/preprocessed_capstone2025.json")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)


Define constants and variables

In [3]:
# Replacements for malformed themes
rthema_corrections = {
    "HandelsrechtGesellschaftsrecht": "Handelsrecht Gesellschaftsrecht",
    "MietePacht": "Miete Pacht"
}

allowed_rthemen = {"Zivilverfahrensrecht", "SchuldrechtAT", "Schadensersatz", "Handelsrecht Gesellschaftsrecht", "BGBAT", "Miete Pacht", "Sachenrecht", "Versicherungsrecht","Kauf Tausch Leasing","Erbschaft Schenkung", "Schuldverhältnisse", "Werkvertrag", "EU-Recht", "Wohnungseigentum", "Sonstiges Recht", "Reisevertrag"}
date_cutoff = "2000-01-01"

json_output = "Data/clean/filtered_output.json"

Save as JSON

In [4]:
# Function to save the cleanded data to JSON, with helper functions

def correct_rthema_list(rthema_list):
    """Apply correction to rthema list items."""
    if not isinstance(rthema_list, list):
        return rthema_list
    return [rthema_corrections.get(r, r) for r in rthema_list]

def rthema_is_valid(rthema_list):
    """Check if corrected rthema is entirely within the allowed set."""
    if not rthema_list or not isinstance(rthema_list, list):
        return False
    return set(rthema_list).issubset(allowed_rthemen)

# Process and filter in-place
filtered_data = {}

for doc_id, content in data.items():
    bibliography = content.get("bibliographische-angaben", {})
    text = content.get("text", {})
    allgemeine = content.get("allgemeine-angaben", {})

    datum_str = bibliography.get("datum", [None])[0]
    try:
        datum = pd.to_datetime(datum_str, format="%Y-%m-%d", errors="coerce")
    except Exception:
        datum = pd.NaT

    rthema_corrected = correct_rthema_list(allgemeine.get("rthema", []))

    if (
        content.get("dokument_art", [None])[0] == "Urteil" and
        content.get("dokument_typ", [None])[0] == "Obere Rechtsprechung" and
        bibliography.get("institution", [None])[0] is not None and
        datum and datum > pd.Timestamp(date_cutoff) and
        rthema_is_valid(rthema_corrected)
    ):
        # Save corrected rthema back into the content
        allgemeine["rthema"] = rthema_corrected
        content["bibliographische-angaben"]["datum"] = [datum.strftime("%Y-%m-%d")]
        filtered_data[doc_id] = content

# Save output JSON
with open(json_output, "w", encoding="utf-8") as f:
    json.dump(filtered_data, f, ensure_ascii=False, indent=2)

Save as CSV

In [ ]:
# Flatten the JSON into a structured DataFrame
records = []
for doc_id, content in data.items():
    base = {
        "doc_id": doc_id.removesuffix(".xml"),
        "schema": content.get("schema"),
        "dokument_art": content.get("dokument_art", [None])[0],
        "dokument_typ": content.get("dokument_typ", [None])[0],
        "institution": content.get("bibliographische-angaben", {}).get("institution", [None])[0],
        "aktenzeichen": content.get("bibliographische-angaben", {}).get("aktenzeichen", [None])[0],
        "datum": pd.to_datetime(content.get("bibliographische-angaben", {}).get("datum", [None])[0], format="%Y-%m-%d", errors="coerce"),
        "fundstelle": "; ".join(content.get("bibliographische-angaben", {}).get("fundstelle_kuerzel", [])),
        "vorinstanz": "; ".join(content.get("bibliographische-angaben", {}).get("vorinstanz", [])),
        "norm_vorinstanz_aktenzeichen": "; ".join(content.get("bibliographische-angaben", {}).get("norm_vorinstanz_aktenzeichen", [])),
        "norm_kuerzel": "; ".join(content.get("bibliographische-angaben", {}).get("norm_kuerzel", [])),
        "titel": content.get("text", {}).get("titel", [None])[0],
        "leitsatz": "; ".join(content.get("text", {}).get("entscheidungsinhalt", {}).get("leitsatz", [])),        
        "gruende": "; ".join([v for k, v in sorted(content.get("text", {}).get("gruende", {}).get("gruende", {}).items(), key=lambda x: int(x[0]))]),
        "rthema": "; ".join(content.get("allgemeine-angaben", {}).get("rthema", [])),
        "quelle": content.get("interne-angaben", {}).get("quelle", [None])[0]
    }
    records.append(base)
df = pd.DataFrame(records)

# Handle misslabled rthemen
def correct_rthema(rthema_str):
    if pd.isna(rthema_str):
        return rthema_str
    for wrong, correct in rthema_corrections.items():
        rthema_str = rthema_str.replace(wrong, correct)
    return rthema_str

def rthema_valid(rthema_string, allowed_set):
    if pd.isna(rthema_string):
        return False
    themen = set(rthema_string.split("; "))
    return themen.issubset(allowed_set)

df["rthema"] = df["rthema"].apply(correct_rthema)

df_filtered = df[
    df["dokument_art"].isin(["Urteil"]) &
    df["dokument_typ"].isin(["Obere Rechtsprechung"]) &
    df["institution"].notna() &
    (df["datum"] > pd.Timestamp(date_cutoff)) &
    df["rthema"].apply(lambda x: rthema_valid(x, allowed_rthemen))
].reset_index(drop=True)

df_filtered.to_csv("Data/clean/filtered_fabi.csv", index=False)